# VisionAssist Phase 7C — Git Clone + Persistent Prepared Data

This notebook:

1. mounts Google Drive;
2. clones or updates the VisionAssist Git repository;
3. restores already prepared data from a single Drive archive;
4. installs the project with `uv`;
5. runs a five-record Qwen2.5-VL baseline smoke test;
6. resumes the full 2,100-record benchmark;
7. stores partial/final outputs in Drive.

Select **Runtime → Change runtime type → GPU** before running.

## One-time preparation on your Windows PC

From the project root:

```powershell
uv run python scripts/prepare_colab_data_bundle.py
```

Upload the generated file:

```text
project_snapshots/visionassist_prepared_data.tar.gz
```

to:

```text
MyDrive/visionassist/data/visionassist_prepared_data.tar.gz
```

The archive contains the prepared VisA images, instruction JSONL files, splits,
benchmarks, manifests, and readiness reports. The repository itself is cloned
from GitHub and is not included in the data archive.

In [ ]:
#@title 1. Settings
from pathlib import Path

# Replace with your real repository URL.
REPO_URL = "https://github.com/<YOUR_USERNAME>/visionassist-industrial-visual-inspection.git"
REPO_BRANCH = "main"

DRIVE_ROOT = Path("/content/drive/MyDrive/visionassist")
DRIVE_DATA_ARCHIVE = DRIVE_ROOT / "data/visionassist_prepared_data.tar.gz"
DRIVE_OUTPUT_ROOT = DRIVE_ROOT / "outputs"
DRIVE_HF_CACHE_ARCHIVE = DRIVE_ROOT / "hf_cache/qwen25vl3b_cache.tar.gz"

PROJECT_ROOT = Path("/content/visionassist-industrial-visual-inspection")
LOCAL_HF_CACHE = Path("/content/hf_cache")

DIRECT_CONFIG = PROJECT_ROOT / "configs/inference/qwen25vl3b_direct.yaml"
LOCAL_DIRECT_OUTPUT = PROJECT_ROOT / "outputs/baseline/qwen2_5_vl_3b_direct"
DRIVE_DIRECT_OUTPUT = DRIVE_OUTPUT_ROOT / "qwen2_5_vl_3b_direct"

SMOKE_STOP_AFTER = 5
USE_4BIT = False
SEED = 42

# Set True only when the Drive archive does not exist and you intentionally want
# Colab to download and prepare the full dataset using Phases 1–7A.
BUILD_DATA_IN_COLAB_IF_MISSING = False

In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")

for directory in (
    DRIVE_ROOT / "data",
    DRIVE_ROOT / "outputs",
    DRIVE_ROOT / "hf_cache",
    DRIVE_ROOT / "logs",
):
    directory.mkdir(parents=True, exist_ok=True)

print("Drive root:", DRIVE_ROOT)
print("Prepared-data archive exists:", DRIVE_DATA_ARCHIVE.is_file())

In [ ]:
#@title 3. Inspect GPU
import platform
import shutil
import subprocess
import torch

assert torch.cuda.is_available(), (
    "No CUDA GPU detected. Choose Runtime → Change runtime type → GPU."
)

gpu = torch.cuda.get_device_properties(0)
print("Python:", platform.python_version())
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(gpu.total_memory / 1024**3, 2))
print("BF16 supported:", torch.cuda.is_bf16_supported())
print("Free disk GiB:", round(shutil.disk_usage('/content').free / 1024**3, 2))
subprocess.run(["nvidia-smi"], check=False)

In [ ]:
#@title 4. Install uv
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "uv"],
    check=True,
)
print("uv installed.")

In [ ]:
#@title 5. Clone or update the Git repository
import shutil
import subprocess

assert "<YOUR_USERNAME>" not in REPO_URL, "Set REPO_URL in the settings cell."

if (PROJECT_ROOT / ".git").is_dir():
    print("Repository already exists. Updating...")
    subprocess.run(
        ["git", "fetch", "origin", REPO_BRANCH],
        cwd=PROJECT_ROOT,
        check=True,
    )
    subprocess.run(
        ["git", "checkout", REPO_BRANCH],
        cwd=PROJECT_ROOT,
        check=True,
    )
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", REPO_BRANCH],
        cwd=PROJECT_ROOT,
        check=True,
    )
else:
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_ROOT)],
        check=True,
    )

commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=PROJECT_ROOT,
    text=True,
).strip()
print("Repository:", PROJECT_ROOT)
print("Commit:", commit)

In [ ]:
#@title 6. Install project dependencies
import os
import subprocess

os.chdir(PROJECT_ROOT)

subprocess.run(
    ["uv", "sync", "--extra", "training"],
    check=True,
)

subprocess.run(
    [
        "uv", "run", "python", "-c",
        (
            "import torch, torchvision, transformers, accelerate, "
            "bitsandbytes; "
            "print('torch', torch.__version__); "
            "print('transformers', transformers.__version__)"
        ),
    ],
    check=True,
)

In [ ]:
#@title 7. Restore prepared data from Google Drive
import tarfile

def prepared_data_present() -> bool:
    required = (
        PROJECT_ROOT / "data/raw/visa",
        PROJECT_ROOT / "data/processed/visa_instructions/test.jsonl",
        PROJECT_ROOT / "data/benchmarks/visa_baseline_v1/benchmark.jsonl",
    )
    return all(path.exists() for path in required)

if prepared_data_present():
    print("Prepared data already exists locally.")
elif DRIVE_DATA_ARCHIVE.is_file():
    print("Extracting prepared data from Drive to local Colab storage...")
    with tarfile.open(DRIVE_DATA_ARCHIVE, "r:gz") as archive:
        archive.extractall(PROJECT_ROOT)
    assert prepared_data_present(), "Prepared-data archive is incomplete."
    print("Prepared data restored successfully.")
elif BUILD_DATA_IN_COLAB_IF_MISSING:
    print("Prepared-data archive is missing. Building data in Colab...")
else:
    raise FileNotFoundError(
        f"Prepared-data archive not found: {DRIVE_DATA_ARCHIVE}\n"
        "Create it locally with scripts/prepare_colab_data_bundle.py and "
        "upload it to Drive, or set BUILD_DATA_IN_COLAB_IF_MISSING=True."
    )

In [ ]:
#@title 8. Optional: build all prepared data in Colab when Drive archive is missing
import subprocess
import tarfile

if not prepared_data_present() and BUILD_DATA_IN_COLAB_IF_MISSING:
    commands = [
        ["uv", "run", "visionassist", "phase1-visa", "--config", "configs/data/visa.yaml"],
        ["uv", "run", "visionassist", "phase2-visa", "--config", "configs/data/visa.yaml"],
        ["uv", "run", "visionassist", "phase3-visa", "--config", "configs/data/visa.yaml"],
        ["uv", "run", "visionassist", "phase4-visa", "--config", "configs/data/visa.yaml"],
        ["uv", "run", "visionassist", "phase5-visa", "--config", "configs/data/visa.yaml"],
        [
            "uv", "run", "visionassist", "phase6-visa",
            "--config", "configs/data/visa.yaml",
            "--processor-smoke-test",
        ],
        [
            "uv", "run", "visionassist", "build-baseline-benchmark",
            "--config", "configs/benchmark/visa_baseline_v1.yaml",
        ],
        [
            "uv", "run", "visionassist", "validate-baseline-benchmark",
            "--config", "configs/benchmark/visa_baseline_v1.yaml",
        ],
    ]

    for command in commands:
        print("Running:", " ".join(command))
        subprocess.run(command, cwd=PROJECT_ROOT, check=True)

    assert prepared_data_present()

    include_paths = [
        "data/raw/visa",
        "data/interim",
        "data/processed",
        "data/splits",
        "data/benchmarks",
        "data/manifests",
        "reports/dataset_audit",
        "reports/training_readiness",
    ]

    DRIVE_DATA_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
    print("Saving prepared-data archive to Drive...")
    with tarfile.open(DRIVE_DATA_ARCHIVE, "w:gz") as archive:
        for relative in include_paths:
            source = PROJECT_ROOT / relative
            if source.exists():
                archive.add(source, arcname=relative)

    print("Saved:", DRIVE_DATA_ARCHIVE)
else:
    print("No Colab data build required.")

In [ ]:
#@title 9. Validate restored benchmark data
import subprocess

subprocess.run(
    [
        "uv", "run", "visionassist", "validate-baseline-benchmark",
        "--config", "configs/benchmark/visa_baseline_v1.yaml",
    ],
    cwd=PROJECT_ROOT,
    check=True,
)

In [ ]:
#@title 10. Restore Hugging Face cache
import os
import shutil
import tarfile

if LOCAL_HF_CACHE.exists():
    shutil.rmtree(LOCAL_HF_CACHE)
LOCAL_HF_CACHE.mkdir(parents=True)

if DRIVE_HF_CACHE_ARCHIVE.is_file():
    print("Restoring Hugging Face cache from Drive...")
    with tarfile.open(DRIVE_HF_CACHE_ARCHIVE, "r:gz") as archive:
        archive.extractall(LOCAL_HF_CACHE)
else:
    print("No model cache archive found; model files will download once.")

os.environ["HF_HOME"] = str(LOCAL_HF_CACHE)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
#@title 11. Restore partial inference outputs
import shutil

LOCAL_DIRECT_OUTPUT.mkdir(parents=True, exist_ok=True)
DRIVE_DIRECT_OUTPUT.mkdir(parents=True, exist_ok=True)

for name in (
    "predictions.partial.jsonl",
    "predictions.jsonl",
    "inference_errors.jsonl",
    "run_manifest.json",
):
    source = DRIVE_DIRECT_OUTPUT / name
    if source.is_file():
        shutil.copy2(source, LOCAL_DIRECT_OUTPUT / name)
        print("Restored:", name)

In [ ]:
#@title 12. Configure five-record smoke inference
import yaml

config = yaml.safe_load(DIRECT_CONFIG.read_text(encoding="utf-8"))
config["stop_after"] = SMOKE_STOP_AFTER
config["load_in_4bit"] = USE_4BIT
config["device_map"] = "auto"
config["precision"] = "auto"
config["seed"] = SEED

generation = config.setdefault("generation", {})
generation.update(
    {
        "do_sample": False,
        "num_beams": 1,
        "max_new_tokens": 256,
        "repetition_penalty": 1.0,
    }
)

if USE_4BIT:
    config["run_id"] = str(config.get("run_id", "baseline")) + "_4bit"
    config["output_dir"] = str(config["output_dir"]) + "_4bit"

DIRECT_CONFIG.write_text(
    yaml.safe_dump(config, sort_keys=False),
    encoding="utf-8",
)
print(yaml.safe_dump(config, sort_keys=False))

In [ ]:
#@title 13. Run five-record smoke test
import subprocess

subprocess.run(
    [
        "uv", "run", "visionassist", "baseline-inference",
        "--config", str(DIRECT_CONFIG),
    ],
    cwd=PROJECT_ROOT,
    check=True,
)

In [ ]:
#@title 14. Inspect and persist smoke predictions
import json
import shutil

partial = LOCAL_DIRECT_OUTPUT / "predictions.partial.jsonl"
rows = [
    json.loads(line)
    for line in partial.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

print("Completed predictions:", len(rows))
for row in rows[-5:]:
    print("\nInstruction:", row["instruction_id"])
    print("Task:", row.get("task_family"))
    print("Prediction:", row.get("prediction"))

assert len(rows) >= SMOKE_STOP_AFTER

for name in (
    "predictions.partial.jsonl",
    "predictions.jsonl",
    "inference_errors.jsonl",
    "run_manifest.json",
):
    source = LOCAL_DIRECT_OUTPUT / name
    if source.is_file():
        shutil.copy2(source, DRIVE_DIRECT_OUTPUT / name)
        print("Synced:", name)

## Full resumable direct baseline

After inspecting the smoke predictions, run the remaining cells.

If Colab disconnects, reopen the notebook and rerun through the output restore
cell. The inference runner skips completed instruction IDs.

In [ ]:
#@title 15. Enable full inference
import yaml

config = yaml.safe_load(DIRECT_CONFIG.read_text(encoding="utf-8"))
config["stop_after"] = None
DIRECT_CONFIG.write_text(
    yaml.safe_dump(config, sort_keys=False),
    encoding="utf-8",
)
print("Full inference enabled.")

In [ ]:
#@title 16. Run or resume all 2,100 direct-baseline records
import subprocess

subprocess.run(
    [
        "uv", "run", "visionassist", "baseline-inference",
        "--config", str(DIRECT_CONFIG),
    ],
    cwd=PROJECT_ROOT,
    check=True,
)

In [ ]:
#@title 17. Validate and sync final predictions
import json
import shutil

final_predictions = LOCAL_DIRECT_OUTPUT / "predictions.jsonl"
assert final_predictions.is_file(), (
    "Inference is incomplete. Rerun the previous cell to resume."
)

rows = [
    json.loads(line)
    for line in final_predictions.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
assert len(rows) == 2100, f"Expected 2100 predictions; found {len(rows)}"
assert len({row["instruction_id"] for row in rows}) == 2100

for name in (
    "predictions.partial.jsonl",
    "predictions.jsonl",
    "inference_errors.jsonl",
    "run_manifest.json",
):
    source = LOCAL_DIRECT_OUTPUT / name
    if source.is_file():
        shutil.copy2(source, DRIVE_DIRECT_OUTPUT / name)

print("Direct baseline complete and synced.")

In [ ]:
#@title 18. Evaluate direct baseline and sync metrics
import shutil
import subprocess

subprocess.run(
    [
        "uv", "run", "visionassist", "evaluate-baseline",
        "--benchmark",
        "data/benchmarks/visa_baseline_v1/benchmark.jsonl",
        "--predictions",
        "outputs/baseline/qwen2_5_vl_3b_direct/predictions.jsonl",
        "--config",
        "configs/evaluation/visa_baseline.yaml",
    ],
    cwd=PROJECT_ROOT,
    check=True,
)

local_eval = PROJECT_ROOT / "outputs/baseline/evaluation"
drive_eval = DRIVE_OUTPUT_ROOT / "direct_evaluation"
shutil.copytree(local_eval, drive_eval, dirs_exist_ok=True)
print("Evaluation synced:", drive_eval)

In [ ]:
#@title 19. Persist model cache for later sessions
import tarfile

if not DRIVE_HF_CACHE_ARCHIVE.is_file():
    print("Saving model cache to Drive. This may take several minutes...")
    DRIVE_HF_CACHE_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
    with tarfile.open(DRIVE_HF_CACHE_ARCHIVE, "w:gz") as archive:
        for path in LOCAL_HF_CACHE.rglob("*"):
            archive.add(path, arcname=path.relative_to(LOCAL_HF_CACHE))
    print("Saved:", DRIVE_HF_CACHE_ARCHIVE)
else:
    print("Persistent model cache already exists.")

## Persistent workflow summary

### Code
Always comes from Git:

```text
git clone / git pull
```

### Prepared data
Stored once in Drive:

```text
MyDrive/visionassist/data/visionassist_prepared_data.tar.gz
```

Extracted into local `/content` storage for each Colab session.

### Model cache
Stored once in Drive:

```text
MyDrive/visionassist/hf_cache/qwen25vl3b_cache.tar.gz
```

### Resumable inference outputs
Stored after every run in:

```text
MyDrive/visionassist/outputs/qwen2_5_vl_3b_direct/
```

This separates version-controlled code from large prepared data and generated
model outputs.